In [1]:
'''
# Install unrar
!apt-get install unrar

# Unzip the rar files
!unrar x "/content/test 1.rar"
!unrar x "/content/train 1.rar"
'''

'\n# Install unrar\n!apt-get install unrar\n\n# Unzip the rar files\n!unrar x "/content/test 1.rar"\n!unrar x "/content/train 1.rar"\n'

In [2]:
# pip install tensorflow opencv-python

In [3]:
# Import the necessary libraries
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import time

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.utils import to_categorical
from tensorflow.data import AUTOTUNE
from tensorflow.keras import layers
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

from scipy.linalg import sqrtm

import cv2

import warnings
warnings.filterwarnings('ignore')

In [4]:
# Load the training metadata
train_df = pd.read_csv('./Train1/processed_metadata_train.csv')

In [5]:
# Check 10 samples
train_df.sample(10)

,new_photo_id,label,original_photo_id,augmented
14609,inside_14610_7KEmynUZXObW5fKdFfOMYQ,inside,7KEmynUZXObW5fKdFfOMYQ,False
22691,outside_22692_jZHuZvW4eEDg1NyjCy9QKg,outside,jZHuZvW4eEDg1NyjCy9QKg,False
614,drink_615_6WZmxQCG7QWc3TgYQVkhXQ,drink,6WZmxQCG7QWc3TgYQVkhXQ,False
5412,food_5413_0P3UBpKRqVAPxVTl2PniQQ,food,0P3UBpKRqVAPxVTl2PniQQ,False
12282,inside_12283_PTR5q4aXK2XPBWzzAsjf3g,inside,PTR5q4aXK2XPBWzzAsjf3g,False
10417,inside_10418_2ztcpjOvLPjx1UdffNKnWQ,inside,2ztcpjOvLPjx1UdffNKnWQ,False
10135,inside_10136_mm3EkqTk7ZhX3Nkem4UmKQ,inside,mm3EkqTk7ZhX3Nkem4UmKQ,False
11244,inside_11245_a4HmefTgMmisFlNQYX0slA,inside,a4HmefTgMmisFlNQYX0slA,False
4324,drink_4325_6lsFauvYkZcQ15CnLZT_ig,drink,6lsFauvYkZcQ15CnLZT_ig,False
20655,outside_20656_lGjjiHdMYVLnp1BqB1KBuw,outside,lGjjiHdMYVLnp1BqB1KBuw,False


In [6]:
# Check info
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   new_photo_id       25000 non-null  object
 1   label              25000 non-null  object
 2   original_photo_id  25000 non-null  object
 3   augmented          25000 non-null  bool  
dtypes: bool(1), object(3)
memory usage: 610.5+ KB


In [7]:
# Load the test metadata
test_df = pd.read_csv('./test/processed_metadata.csv')

In [8]:
# Check 10 samples
test_df.sample(10)

,new_photo_id,label,original_photo_id,augmented
1057,food_1058_aRACVOPiPDbxKXNmpMWtIQ,food,aRACVOPiPDbxKXNmpMWtIQ,False
3961,menu_3962_aug,menu,MZAnR2vEC7i3mfLqQ5oQ5g,True
2661,inside_2662_35NZwnJ1SOCevTtOGSlHag,inside,35NZwnJ1SOCevTtOGSlHag,False
3212,menu_3213_aug,menu,Nfr0X6Zvo-IXiixELtcSCA,True
975,drink_976_6P6yZ2IkoAT5Wma4ZRUAhQ,drink,6P6yZ2IkoAT5Wma4ZRUAhQ,False
2301,inside_2302_1-9VkFVRs0guskqefBeDhA,inside,1-9VkFVRs0guskqefBeDhA,False
3744,menu_3745_aug,menu,BaewJqGSrbvBawkQyGYWhw,True
4627,outside_4628_JvRqwjRg69k6Ybbr4RzcZg,outside,JvRqwjRg69k6Ybbr4RzcZg,False
1053,food_1054_scPZ6nhddcQy40UE3EZtXw,food,scPZ6nhddcQy40UE3EZtXw,False
3033,menu_3034_13C7G8tHfu8N3BFxvt_Qug,menu,13C7G8tHfu8N3BFxvt_Qug,False


In [9]:
# Check info
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   new_photo_id       5000 non-null   object
 1   label              5000 non-null   object
 2   original_photo_id  5000 non-null   object
 3   augmented          5000 non-null   bool  
dtypes: bool(1), object(3)
memory usage: 122.2+ KB


## Filter Valid Photos

In [10]:
# Remove all datasets with "Menu" label
train_df = train_df[train_df['label'] != 'menu'].reset_index(drop=True)

In [11]:
# Remove all datasets with "Menu" label
test_df = test_df[test_df['label'] != 'menu'].reset_index(drop=True)

In [12]:
# Image directory
img_dir_train = './Train1/processed_photos_train'
img_dir_test = './test/processed_photos'

In [13]:
# Count the items in image directories
num_items_train = len(os.listdir(img_dir_train))
num_items_test = len(os.listdir(img_dir_test))

print(f"Number of items in {img_dir_train}: {num_items_train}")
print(f"Number of items in {img_dir_test}: {num_items_test}")

Number of items in ./Train1/processed_photos_train: 25000
Number of items in ./test/processed_photos: 5000


In [14]:
def filter_valid_photos(df, image_dir):
    valid_indices = []
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Validating photos"):
        photo_id = row['new_photo_id']
        img_path = os.path.join(image_dir, f'{photo_id}.jpg')
        try:
            _ = tf.keras.preprocessing.image.load_img(img_path)
            valid_indices.append(i)
        except Exception as e:
            continue
    return df.loc[valid_indices].reset_index(drop=True)

# Clean metadata
start_time = time.time()
train_df = filter_valid_photos(train_df, image_dir=img_dir_train)
test_df = filter_valid_photos(test_df, image_dir=img_dir_test)
print(f'After cleaning, training metadata has {len(train_df)} rows. Took {time.time()-start_time:.2f}s.')
print(f'After cleaning, test metadata has {len(test_df)} rows. Took {time.time()-start_time:.2f}s.')

Validating photos: 100%|█████████████████████████████████████████████████████████| 4000/4000 [00:03<00:00, 1051.53it/s]

After cleaning, training metadata has 20000 rows. Took 25.76s.
After cleaning, test metadata has 4000 rows. Took 25.76s.


In [15]:
train_df.head()

,new_photo_id,label,original_photo_id,augmented
0,drink_1_1u4I3V3fhRDDfLHRbDwO9w,drink,1u4I3V3fhRDDfLHRbDwO9w,False
1,drink_2_kCob6wOKqwm6hXQ-WLYCOA,drink,kCob6wOKqwm6hXQ-WLYCOA,False
2,drink_3_0eDzm3uqdjvj1srynbO0uw,drink,0eDzm3uqdjvj1srynbO0uw,False
3,drink_4_tOATeq906QAXmmB8jWouNw,drink,tOATeq906QAXmmB8jWouNw,False
4,drink_5_Vc0E0fGqglH0Bj8fpfR29g,drink,Vc0E0fGqglH0Bj8fpfR29g,False


In [16]:
test_df.head()

,new_photo_id,label,original_photo_id,augmented
0,drink_1_8VfJq1vTpMh6IMKhtbsqHg,drink,8VfJq1vTpMh6IMKhtbsqHg,False
1,drink_2_3LGq6soWgPCWeVqaJCqCnw,drink,3LGq6soWgPCWeVqaJCqCnw,False
2,drink_3_YoIAYWkm-2Di1c46gyJUXQ,drink,YoIAYWkm-2Di1c46gyJUXQ,False
3,drink_4_NwjHP2DiGj3xhCd8PnYcgg,drink,NwjHP2DiGj3xhCd8PnYcgg,False
4,drink_5_1XR1orXOs9DcwWwbep9USw,drink,1XR1orXOs9DcwWwbep9USw,False


# 4. cGAN

## Encode labels: [food, drink, inside, outside]

In [17]:
# Initialize labels
LABELS = ['food', 'drink', 'inside', 'outside']

In [18]:
le = LabelEncoder()
train_df['label_encoded'] = le.fit_transform(train_df['label'])
test_df['label_encoded'] = le.fit_transform(test_df['label'])

encoder = OneHotEncoder(sparse_output=False)
onehot_labels_train = encoder.fit_transform(train_df[['label_encoded']])
onehot_labels_test = encoder.transform(test_df[['label_encoded']])

print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

# Test small sample
sample_train_df = train_df.sample(5)
sample_test_df = test_df.sample(5)
print(sample_train_df[['new_photo_id', 'label', 'label_encoded']])
print('\n')
print(sample_test_df[['new_photo_id', 'label', 'label_encoded']])

Label mapping: {'drink': 0, 'food': 1, 'inside': 2, 'outside': 3}
                               new_photo_id    label  label_encoded
10180   inside_10181_97G_WxIhKNEa9nzHg2qXZA   inside              2
4987      drink_4988_5BMsmPxBfiBbnVuhRdvfuw    drink              0
14076   inside_14077_pBBEhdZmpEVqocQL7gI_BA   inside              2
15037  outside_20038_apnbpB3lxnnarUy6rWfLwg  outside              3
10181   inside_10182_cNEDUtY3yUOj-QN55pcLAw   inside              2


                             new_photo_id    label  label_encoded
449      drink_450_4cG2q83zrQjiMw3Q8A-qkw    drink              0
1471     food_1472_fRah4IcH-2yeJ-NHcQfIaA     food              1
252      drink_253_00j6oA3hT357an8p_q7fOQ    drink              0
2516   inside_2517__fQgFLIqaV0aNWrw7-pfhw   inside              2
3010  outside_4011_rrskVR7yHHJLlh6gsELloQ  outside              3


## Load sample images to test pipeline

In [19]:
def load_images(photo_ids, directory=img_dir_train, target_size=(64,64)):
    images = []
    for pid in tqdm(photo_ids, desc="Loading images"):
        try:
            path = os.path.join(directory, f'{pid}.jpg')
            img = tf.keras.preprocessing.image.load_img(path, target_size=target_size)
            img = tf.keras.preprocessing.image.img_to_array(img) / 255.0
            images.append(img)
        except Exception as e:
            print(f'Could not load {path}: {e}')
            continue
    return np.array(images)

# Load a small sample for quick test
start_time = time.time()
train_sample = load_images(train_df['new_photo_id'].sample(5))
test_sample = load_images(test_df['new_photo_id'].sample(5), directory=img_dir_test)
print(f'Loaded {len(train_sample)} sample images in {time.time()-start_time:.2f}s')
print(f'Loaded {len(test_sample)} sample images in {time.time()-start_time:.2f}s')

Loading images: 100%|██████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 1002.61it/s]

Loaded 5 sample images in 0.03s
Loaded 5 sample images in 0.03s


## Load images from the main (Training and Test) dataframes

In [20]:
start_time = time.time()
train_array = load_images(train_df['new_photo_id'])
test_array = load_images(test_df['new_photo_id'], directory=img_dir_test)
print(f'Loaded {len(train_array)} total images in {time.time()-start_time:.2f}s')
print(f'Loaded {len(test_array)} total images in {time.time()-start_time:.2f}s')

Loading images: 100%|████████████████████████████████████████████████████████████| 4000/4000 [00:02<00:00, 1665.78it/s]


Loaded 20000 total images in 15.59s
Loaded 4000 total images in 15.59s


In [21]:
# Create arrays and specify labels
valid_indices_train = [i for i, img in enumerate(train_array)]
valid_indices_test = [i for i, img in enumerate(test_array)]
all_images_train = np.array(train_array)
all_images_test = np.array(test_array)
all_labels_train = onehot_labels_train[train_df['label_encoded']]
all_labels_test = onehot_labels_test[test_df['label_encoded']]

print('Final train dataset shape:', all_images_train.shape, all_labels_train.shape)
print('Final test dataset shape:', all_images_test.shape, all_labels_test.shape)

Final train dataset shape: (20000, 64, 64, 3) (20000, 4)
Final test dataset shape: (4000, 64, 64, 3) (4000, 4)


# Build cGAN

In [22]:
# Enable GPU optimizations
physical_devices = tf.config.experimental.list_physical_devices("GPU")
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f"Using GPU: {physical_devices[0]}")
else:
    print("No GPU found. Using CPU.")

No GPU found. Using CPU.


## Generator

In [23]:
def make_generator():
    noise_input = layers.Input(shape=(100,))
    label_input = layers.Input(shape=(4,))
    
    # Concatenate noise and label
    x = layers.Concatenate()([noise_input, label_input])
    
    # Added use_bias=False for better performance
    x = layers.Dense(8*8*256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    x = layers.Reshape((8,8,256))(x)

    # Added use_bias=False to Conv2DTranspose layers
    x = layers.Conv2DTranspose(128, (4,4), strides=(2,2), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2DTranspose(64, (4,4), strides=(2,2), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    # Final layer with tanh activation
    x = layers.Conv2DTranspose(3, (4,4), strides=(2,2), padding='same', activation='tanh')(x)
    
    return tf.keras.Model([noise_input, label_input], x, name='Generator')

## Discriminator

In [24]:
def make_discriminator():
    img_input = layers.Input(shape=(64,64,3))
    label_input = layers.Input(shape=(4,))
    
    # Expand label to match image dimensions
    label_expanded = layers.Dense(64*64*1)(label_input)
    label_expanded = layers.Reshape((64,64,1))(label_expanded)
    
    # Concatenate image and label
    x = layers.Concatenate()([img_input, label_expanded])
    
    # Convolutional layers
    x = layers.Conv2D(64, (4,4), strides=(2,2), padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, (4,4), strides=(2,2), padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv2D(256, (4,4), strides=(2,2), padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(1)(x)
    
    return tf.keras.Model([img_input, label_input], x, name='Discriminator')

## Training

In [25]:
# Initialize models
generator = make_generator()
discriminator = make_discriminator()

# Using BinaryCrossentropy with from_logits=True and optimized loss functions
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

In [36]:
# Optimized learning rates and beta_1 parameter
gen_opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
disc_opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

# Optimized training step with better gradient handling
@tf.function
def train_step(images, labels):
    noise = tf.random.normal([images.shape[0], 100])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator([noise, labels], training=True)

        real_output = discriminator([images, labels], training=True)
        fake_output = discriminator([generated_images, labels], training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    gen_opt.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    disc_opt.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

In [37]:
# Image generator function
def generate_sample(label_text, ax, generator, le, encoder):
    idx = le.transform([label_text])
    onehot = encoder.transform(idx.reshape(-1,1))
    onehot = tf.convert_to_tensor(onehot, dtype=tf.float32)
    noise = tf.random.normal([1,100])
    gen_img = generator([noise, onehot], training=False)
    ax.imshow((gen_img[0]+1)/2)
    ax.set_title(label_text)
    ax.axis('off')

In [38]:
# Constant
AUTOTUNE = tf.data.AUTOTUNE

# Optimized dataset pipeline
def create_optimized_dataset(images, labels, batch_size=64):
    dataset = tf.data.Dataset.from_tensor_slices((images, labels))
    dataset = dataset.shuffle(10000)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset

## Training loop

In [39]:
# Main training function with all features
def train_complete_cgan(all_images_train, all_labels_train, epochs=50):
    # Create optimized dataset
    dataset = create_optimized_dataset(all_images_train, all_labels_train)
    
    # Training loop with progress tracking
    for epoch in range(epochs):
        start_epoch = time.time()
        epoch_g_loss = []
        epoch_d_loss = []
        
        # Enhanced progress bar
        progress_bar = tqdm(dataset, desc=f"Epoch {epoch+1}/{epochs}", leave=True)
        
        for batch_images, batch_labels in progress_bar:
            g_loss, d_loss = train_step(batch_images, batch_labels)
            epoch_g_loss.append(g_loss.numpy())
            epoch_d_loss.append(d_loss.numpy())
            
            # Update progress bar
            progress_bar.set_postfix(
                G_Loss=f"{g_loss.numpy():.4f}", 
                D_Loss=f"{d_loss.numpy():.4f}"
            )

        end_epoch = time.time()
        avg_g_loss = np.mean(epoch_g_loss)
        avg_d_loss = np.mean(epoch_d_loss)
        
        print(f'Epoch {epoch+1}/{epochs}: Gen loss={avg_g_loss:.4f}, Disc loss={avg_d_loss:.4f} '
              f'in {end_epoch-start_epoch:.2f}s')

        # Generate sample images every 5 epochs
        if (epoch+1) % 5 == 0:
            print(f'Showing generated samples at epoch {epoch+1}')
            fig, axes = plt.subplots(1, 4, figsize=(12,3))
            for i, label in enumerate(LABELS):
                generate_sample(label, axes[i], generator, le, encoder)
            plt.show()

## Evaluation of cGAN

In [40]:
# IS and FID calculation
def calculate_is_fid_optimized(real_images, fake_images):
    
    # Convert tensors to NumPy arrays
    real_images = real_images.numpy() if tf.is_tensor(real_images) else real_images
    fake_images = fake_images.numpy() if tf.is_tensor(fake_images) else fake_images

    # Compute statistics for IS (simplified version)
    p_yx = tf.nn.softmax(fake_images.reshape(fake_images.shape[0], -1))
    p_y = tf.reduce_mean(p_yx, axis=0)
    kl_div = tf.reduce_sum(p_yx * tf.math.log(p_yx / (p_y + 1e-10) + 1e-10), axis=1)
    inception_score = tf.exp(tf.reduce_mean(kl_div))

    # Compute FID with dimensionality reduction
    mu_real = np.mean(real_images, axis=(0, 1, 2))
    mu_fake = np.mean(fake_images, axis=(0, 1, 2))

    # Flatten for covariance computation
    real_images_flat = real_images.reshape(real_images.shape[0], -1)
    fake_images_flat = fake_images.reshape(fake_images.shape[0], -1)
    
    sigma_real = np.cov(real_images_flat, rowvar=False)
    sigma_fake = np.cov(fake_images_flat, rowvar=False)

    # Compute FID
    diff = mu_real - mu_fake
    sqrt_cov = sqrtm(sigma_real @ sigma_fake)

    if np.iscomplexobj(sqrt_cov):
        sqrt_cov = sqrt_cov.real

    fid_score = np.linalg.norm(diff) + np.trace(sigma_real + sigma_fake - 2 * sqrt_cov)

    return inception_score.numpy(), fid_score


In [41]:
# FID computation with InceptionV3
def compute_fid_multisample(real_imgs, fake_imgs, batch_size=32):

    # Ensure images are float32
    real_imgs = tf.convert_to_tensor(real_imgs, dtype=tf.float32)
    fake_imgs = tf.convert_to_tensor(fake_imgs, dtype=tf.float32)

    # Resize to (299,299,3) for Inception
    real_imgs_resized = tf.image.resize(real_imgs, (299,299)).numpy()
    fake_imgs_resized = tf.image.resize(fake_imgs, (299,299)).numpy()

    # Preprocess for InceptionV3
    real_imgs_resized = preprocess_input(real_imgs_resized)
    fake_imgs_resized = preprocess_input(fake_imgs_resized)

    # Load Inception model
    model = InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))

    # Get activations
    act_real = model.predict(real_imgs_resized, batch_size=batch_size, verbose=0)
    act_fake = model.predict(fake_imgs_resized, batch_size=batch_size, verbose=0)

    # Compute statistics
    mu1, sigma1 = act_real.mean(axis=0), np.cov(act_real, rowvar=False)
    mu2, sigma2 = act_fake.mean(axis=0), np.cov(act_fake, rowvar=False)

    # FID calculation
    ssdiff = np.sum((mu1 - mu2)**2)
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = ssdiff + np.trace(sigma1 + sigma2 - 2*covmean)
    
    print(f"FID Score: {fid:.4f}")
    return fid

In [42]:
# Plot distributions function
def plot_distributions(real_images, fake_images, real_labels, fake_labels, discriminator):

    # Flatten images for visualization
    real_images_flat = tf.reshape(real_images, [-1])
    fake_images_flat = tf.reshape(fake_images, [-1])

    plt.figure(figsize=(12, 5))

    # Plot pixel distributions
    plt.subplot(1, 2, 1)
    sns.histplot(real_images_flat.numpy(), bins=50, color='blue', label='Real', kde=True, stat="density")
    sns.histplot(fake_images_flat.numpy(), bins=50, color='red', label='Generated', kde=True, stat="density")
    plt.title("Pixel Value Distribution: Real vs Generated")
    plt.legend()

    # Get discriminator scores
    real_scores = discriminator([real_images, real_labels])
    fake_scores = discriminator([fake_images, fake_labels])

    # Plot discriminator scores
    plt.subplot(1, 2, 2)
    sns.kdeplot(real_scores.numpy().flatten(), color='blue', label="Real Images")
    sns.kdeplot(fake_scores.numpy().flatten(), color='red', label="Generated Images")
    plt.title("Discriminator Score Distribution")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [43]:
# Generate best images based on discriminator confidence
def generate_best_images_multi(generator, discriminator, labels_list, encoder, num_samples=500, top_k=1):

    plt.figure(figsize=(5*len(labels_list), 5))

    for i, label_name in enumerate(labels_list):
        # Encode label to onehot
        idx = encoder.transform([[label_name]])
        onehot = tf.convert_to_tensor(idx.repeat(num_samples, axis=0), dtype=tf.float32)

        # Generate images
        noise = tf.random.normal([num_samples, 100])
        fake_imgs = generator([noise, onehot], training=False)

        # Get discriminator scores
        disc_scores = discriminator([fake_imgs, onehot], training=False).numpy().flatten()

        # Get top-k images
        best_indices = np.argsort(-disc_scores)[:top_k]
        best_img = (fake_imgs[best_indices[0]].numpy() + 1) / 2

        plt.subplot(1, len(labels_list), i+1)
        plt.imshow(best_img)
        plt.title(f"'{label_name}'\nDisc: {disc_scores[best_indices[0]]:.2f}")
        plt.axis('off')

    plt.suptitle(f"Top {top_k} image per category by discriminator confidence")
    plt.tight_layout()
    plt.show()

In [44]:
# Complete evaluation pipeline with all metrics and visualizations
def complete_evaluation(generator, discriminator, dataset, encoder, num_images=1000):

    print(" Starting Complete Evaluation...")
    
    # Generate fake images for evaluation
    noise = tf.random.normal([num_images, 100])
    labels = tf.one_hot(np.random.choice(4, num_images), depth=4)
    generated_images = generator([noise, labels], training=False)

    # Get real images
    real_images, real_labels = next(iter(dataset.take(1)))
    real_images = real_images[:num_images]
    real_labels = real_labels[:num_images]

    # Compute IS and FID scores
    print("Computing IS and FID scores...")
    is_score, fid_score = calculate_is_fid_optimized(real_images, generated_images)
    
    # Plot IS and FID scores
    plt.figure(figsize=(8, 4))
    plt.bar(["Inception Score", "FID Score"], [is_score, fid_score], color=['blue', 'red'])
    plt.ylabel("Score")
    plt.title("Inception Score vs FID Score")
    plt.show()

    print(f"Inception Score: {is_score:.4f}")
    print(f"FID Score: {fid_score:.4f}")

    # Plot distributions
    print("Plotting distributions...")
    plot_distributions(real_images, generated_images, real_labels, labels, discriminator)

    # Show generated images with labels
    print("Showing sample generated images...")
    num_images_to_show = 5
    plt.figure(figsize=(15, 3))

    for i in range(num_images_to_show):
        plt.subplot(1, num_images_to_show, i + 1)
        plt.imshow((generated_images[i] * 0.5) + 0.5)
        plt.title(f"Label: {labels[i].numpy()}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

    # Generate best images
    print("Generating best images by discriminator confidence...")
    generate_best_images_multi(generator, discriminator, LABELS, encoder, num_samples=500, top_k=1)

    # Save generated samples
    save_dir = "generated_samples"
    os.makedirs(save_dir, exist_ok=True)

    def generate_save_plot_images():
        noise = tf.random.normal([4, 100])
        sample_labels = tf.eye(4)
        generated_images = generator([noise, sample_labels], training=False)

        fig, axs = plt.subplots(1, 4, figsize=(12, 3))
        for i, img in enumerate(generated_images):
            img = (img + 1) / 2
            plt.imsave(f"{save_dir}/{LABELS[i]}.png", img.numpy())
            axs[i].imshow(img)
            axs[i].axis("off")
            axs[i].set_title(LABELS[i])
        
        print(f"Images saved in {save_dir}")
        plt.tight_layout()
        plt.show()

    generate_save_plot_images()

In [ ]:
# Train the complete model
train_complete_cgan(all_images_train, all_labels_train, epochs=50)

Epoch 1/50: 100%|██████████████████████████████████████| 313/313 [09:58<00:00,  1.91s/it, D_Loss=0.7353, G_Loss=1.2229]


Epoch 1/50: Gen loss=1.1714, Disc loss=0.9664 in 598.71s


Epoch 2/50:  14%|█████▎                                 | 43/313 [01:01<07:04,  1.57s/it, D_Loss=0.8716, G_Loss=1.3138]

In [ ]:
# Run complete evaluation
dataset = create_optimized_dataset(all_images_test, all_labels_test)
complete_evaluation(generator, discriminator, dataset, encoder)